Notebook is used to work with Deployment scripts using PowerShell and Azure CLI when needed.  If you would like to use Cloud Shell in Notebook see ref:  [Title](https://github.com/dotnet/interactive/blob/main/samples/notebooks/powershell/Docs/Interact-With-Azure-Cloud-Shell.ipynb)

In [1]:
#Login to Azure
#Use -Environment Parm to if you need to use Azure Gov.  AzureUSGovernmnet

Connect-AzAccount -Environment AzureUSGovernment


To override which subscription Connect-AzAccount selects by default, use `Update-AzConfig -DefaultSubscriptionForLogin 00000000-0000-0000-0000-000000000000`. Go to https://go.microsoft.com/fwlink/?linkid=2200610 for more information.

Account                                    SubscriptionName                               TenantId
-------                                    ----------------                               -------- 
frgarofa@fasttrackforazure.onmicrosoft.com Azure FFL Internal Subscription FRGAROFA (GOV) 3666803f…



In [2]:
#Select Subscription

$subscription = "Azure FFL Internal Subscription FRGAROFA (GOV)"
Select-AzSubscription -Subscription $subscription



Name                                     Account        SubscriptionNa Environment    TenantId
                                                        me
----                                     -------        -------------- -----------    --------
Azure FFL Internal Subscription FRGAROF… frgarofa@fast… Azure FFL Int… AzureUSGovern… 3666803f-d1e…



In [9]:
#Check Resource Group 

$ResourceGroup = 'dmz-cloudscale-dev'

Get-AzResource -ResourceGroupName $ResourceGroup | Format-Table -AutoSize


Name             ResourceGroupName  ResourceType                  Location
----             -----------------  ------------                  --------
dmz-eventhub-dev dmz-cloudscale-dev Microsoft.EventHub/namespaces usgovvirginia



In [26]:
dir


    Directory: S:\Repos\GitHub\csa-inabox\deploy\notebooks

Mode                 LastWriteTime         Length Name
----                 -------------         ------ ----
-a---           7/19/2023  2:15 PM          11894 PowerShellHelperNotebook.ipynb



In [27]:
#Set Param values for deplkoyment template

$params = @{
    ResourceGroupName = 'dmz-cloudscale-dev'
    TemplateFile = '..\arm\DMZ\Purview\Purview.template.json'
    TemplateParameterFile = '..\arm\DMZ\Purview\Purview.parameters.json'
    Verbose = $true
}

In [28]:
#Test ARM deplkoyment
Test-AzResourceGroupDeployment @params


VERBOSE: 2:21:12 PM - Template is valid.


In [29]:
#Deploy ARM Template

New-AzResourceGroupDeployment @params

VERBOSE: Performing the operation "Creating Deployment" on target "dmz-cloudscale-dev".
VERBOSE: 2:21:28 PM - Template is valid.
VERBOSE: 2:21:29 PM - Create template deployment 'Purview.template'
VERBOSE: 2:21:29 PM - Checking deployment status in 5 seconds
New-AzResourceGroupDeployment: 
Line |
   3 |  New-AzResourceGroupDeployment @params
     |  ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
     | 2:21:34 PM - The deployment 'Purview.template' failed with error(s). Showing 1 out of 1 error(s).
Status Message: The resource type could not be found in the namespace 'Microsoft.Purview' for api version '2020-12-01-preview'. (Code:InvalidResourceType)

CorrelationId: f5399086-5a79-45d2-a404-b52fde850995

DeploymentName          : Purview.template
ResourceGroupName       : dmz-cloudscale-dev
ProvisioningState       : Failed
Timestamp               : 7/19/2023 6:21:33 PM
Mode                    : Incremental
TemplateLink            : 
Parameters              : 
                          Name      

In [20]:
#Get list of Supported API versions for resource ProviderNamespace

Get-AzResourceProvider -ProviderNamespace Microsoft.Purview | Select-Object ProviderNamespace -ExpandProperty ResourceTypes | Select-Object * -ExpandProperty ApiVersions | Sort-Object -Descending

2021-12-01
2021-12-01
2021-12-01
2021-12-01
2021-12-01
2021-12-01
2021-07-01
2021-07-01
2021-07-01
2021-07-01
2021-07-01
2021-07-01
2020-12-01-preview
2020-12-01-preview
2020-12-01-preview
2020-12-01-preview
2020-12-01-preview
2020-12-01-preview


In [19]:
#Set default API version for Purview

$token = Get-AzAccessToken -Resource 'https://management.usgovcloudapi.net/'
$bearer = $token.Token

$params = @{
    Uri     = 'https://management.usgovcloudapi.net/subscriptions/02cd7474-626e-4aa4-b21f-4856bdb92779/providers/microsoft.purview/register?api-version=2021-07-01-preview'
    Method  = 'Post'
    Headers = @{'Authorization' = "Bearer $($bearer)" }
    Body    = '{"resourcetype": "Microsoft.Purview/accounts","location":["usgovvirginia"],"apiVersions": ["2021-12-01","2021-07-01","2020-12-01-preview"],"defaultApiVersion": "2019-06-01","capabilities": "None"
    }'
   }

Invoke-RestMethod @params



id                 : /subscriptions/02cd7474-626e-4aa4-b21f-4856bdb92779/providers/Microsoft.Purvie
                     w
namespace          : Microsoft.Purview
authorizations     : {@{applicationId=73c2949e-da2d-457a-9607-fcc665198967; 
                     roleDefinitionId=1BC09725-0C9B-4F57-A3D0-FCCF4EB40120; 
                     managedByRoleDefinitionId=9e3af657-a8ff-583c-a75c-2fe7c4bcb635}, 
                     @{applicationId=fd642066-7bfc-4b65-9463-6a08841c12f0; 
                     roleDefinitionId=5b3958cb-0439-42e1-80e3-cae077d0d5e8}}
resourceTypes      : {@{resourceType=operations; locations=System.Object[]; 
                     apiVersions=System.Object[]; capabilities=None}, 
                     @{resourceType=setDefaultAccount; locations=System.Object[]; 
                     apiVersions=System.Object[]; defaultApiVersion=2021-07-01; 
                     capabilities=None}, @{resourceType=removeDefaultAccount; 
                     locations=System.Object[]; apiV